# Re-run of the lookahead sweep with end-to-end lookahead

The first arXiv version applied the lookahead mask in every encoder layer and
every probe block, so right context compounded with depth: a labelled 40 ms
reached at least 220 ms of future audio (`eval/effective_lookahead.py`). This
notebook repeats every training experiment in the paper with
`--lookahead-scope end_to_end`, where the encoder is strictly causal and only
the first probe block looks ahead, so future context is exactly chunk + L.

Each sweep appends to a `.jsonl` on Drive after every run and takes
`--resume`, so a reclaimed session loses at most one run: re-run cells 1-4,
then the cell that was interrupted. Rough T4 time for everything: 8-9 hours.

In [ ]:
#@title 1. GPU
import torch, json
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU, then Restart.'
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
#@title 2. Clone the repo and mount Drive
GIT_URL = 'https://github.com/Dipeshtripathi13/streaming-fac-lookahead.git'  #@param {type:'string'}
BRANCH = 'end-to-end-lookahead'  #@param {type:'string'}
import os
REPO = '/content/research'
!rm -rf {REPO} && git clone -q -b {BRANCH} {GIT_URL} {REPO} && echo cloned {BRANCH}
from google.colab import drive; drive.mount('/content/drive')
RES = '/content/drive/MyDrive/sfac_e2e'
os.makedirs(f'{RES}/hyps', exist_ok=True)
%cd {REPO}

In [ ]:
#@title 3. Install and Hugging Face auth (L2-ARCTIC is gated)
!pip -q install soundfile transformers datasets huggingface_hub 2>&1 | tail -2
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title 4. Gates: all must pass before spending GPU time
%cd {REPO}
!python tests/test_causal.py | tail -1
!python tests/test_pipeline_invariants.py | tail -2
# measured future context under both scopes; end_to_end must be exact on every row
!python eval/effective_lookahead.py --out {RES}/effective_lookahead_wavlm.json
!python train/train_translator.py --smoke --device cuda --out {RES}/smoke | tail -4

## A. Main results (WavLM base+)

In [ ]:
#@title 5. Dense sweep, canonical-phone arm: 21 budgets x 2 seeds (~2 h)
!python train/train_translator.py --resume --lookaheads 0 20 40 60 80 100 120 140 160 180 200 220 240 260 280 300 320 340 360 480 640 \
    --targets native --seeds 1337 7 --steps 1200 --device cuda \
    --save-hyps --hyp-dir {RES}/hyps --out {RES}/translator_e2e_dense

In [ ]:
#@title 6. Dense sweep, realized-phone arm: 16 budgets x 2 seeds (~1.5 h)
!python train/train_translator.py --resume --no-verify-causality --lookaheads 0 20 40 60 80 100 120 140 160 180 200 240 280 320 480 640 \
    --targets produced --seeds 1337 7 --steps 1200 --device cuda \
    --save-hyps --hyp-dir {RES}/hyps_produced --out {RES}/translator_e2e_dense_produced

In [ ]:
#@title 7. Three-seed sweep, both arms: 7 budgets x 2 targets x 3 seeds (~2 h)
!python train/train_translator.py --resume --no-verify-causality \
    --lookaheads 0 20 40 80 160 320 640 --targets native produced \
    --seeds 1337 7 99 --steps 1200 --device cuda \
    --save-hyps --hyp-dir {RES}/hyps_3seed --out {RES}/translator_e2e_3seed

In [ ]:
#@title 8. Repeatability floor: 4 budgets x 4 repeats at one seed (~45 min)
# Same seed, same configuration, four complete repetitions. No --resume here:
# the tags are identical by design, so each repeat writes its own file.
import os
for rep in range(1, 5):
    out = f'{RES}/floor_e2e_rep{rep}'
    if os.path.exists(out + '_summary.json'):
        print('repeat', rep, 'already done'); continue
    !rm -f {out}.jsonl
    !python train/train_translator.py --no-verify-causality --lookaheads 40 160 240 320 --targets native --seeds 1337 --steps 1200 --device cuda --out {out} | grep -E "step 1200|wrote"

In [ ]:
#@title 9. Convergence check: 5 budgets to 6000 steps (~1 h)
!python train/train_translator.py --resume --no-verify-causality \
    --lookaheads 0 40 160 240 640 --targets native --seeds 1337 \
    --steps 6000 --eval-every 500 --device cuda --out {RES}/translator_e2e_convergence

## B. Second encoder (wav2vec 2.0 base)

Under end-to-end lookahead both encoders are strictly causal and the lookahead
sits in the probe, so the tap depth no longer changes how much future a label
buys. That removes the confound in the first version's comparison (layer 9
against layer 2). The layer is still chosen by function, at L = 0 only.

In [ ]:
#@title 10. wav2vec 2.0: gate, then layer selection at L = 0 (~15 min)
!python eval/effective_lookahead.py --encoder-name facebook/wav2vec2-base --encoder-layer 2 --out {RES}/effective_lookahead_w2v2.json
for layer in (2, 4, 6, 8, 9):
    !python train/train_translator.py --resume --no-verify-causality --encoder-name facebook/wav2vec2-base --encoder-layer {layer} --lookaheads 0 --targets native --seeds 1337 --steps 1200 --device cuda --out {RES}/w2v2_e2e_layerprobe_l{layer} | grep -E "step 1200|wrote"

In [ ]:
#@title 11. wav2vec 2.0 sweep at the selected layer (~1.2 h)
W2V2_LAYER = 2  #@param {type:'integer'}
E = f'--encoder-name facebook/wav2vec2-base --encoder-layer {W2V2_LAYER} --targets native --device cuda'
!python train/train_translator.py --resume {E} --lookaheads 0 20 40 80 160 320 640 \
    --seeds 1337 --steps 1200 --out {RES}/w2v2_e2e_l{W2V2_LAYER}
!python train/train_translator.py --resume --no-verify-causality {E} --lookaheads 0 40 160 340 640 \
    --seeds 7 --steps 1200 --out {RES}/w2v2_e2e_l{W2V2_LAYER}_seed7
!python train/train_translator.py --resume --no-verify-causality {E} --lookaheads 0 160 640 \
    --seeds 1337 --steps 6000 --eval-every 500 --out {RES}/w2v2_e2e_l{W2V2_LAYER}_convergence

## C. Read the result

In [ ]:
#@title 12. Old (per-layer) against new (end-to-end) curve
import json, collections
def curve(path, target='native'):
    acc = collections.defaultdict(list)
    for line in open(path):
        r = json.loads(line)
        if r['target'] == target: acc[r['lookahead_ms']].append(r['test_per'])
    return {L: sum(v)/len(v) for L, v in sorted(acc.items())}
old = curve(f'{REPO}/results/raw/translator_dense_plus_sat.jsonl')
new = curve(f'{RES}/translator_e2e_dense.jsonl')
print(f"{'L (ms)':>7} {'per-layer':>10} {'end-to-end':>11}")
for L in new:
    print(f"{L:>7g} {old.get(L, float('nan')):>10.4f} {new[L]:>11.4f}")

When everything has finished, copy the Drive folder `sfac_e2e/` into
`results/raw/e2e/` in the repo. The analysis scripts in `eval/` take the
`.jsonl` paths as arguments and need no GPU.

## D. Lookahead inside the encoder (`--lookahead-scope encoder_first`)

Same exact budget, but the lookahead sits in the first encoder layer and every
later layer (encoder and probe) is chunk-causal, so the pretrained encoder
builds its representation with the future in view without the context
compounding. Compared with section A this separates "the encoder needs the
future" from "the old model just had more context".

In [ ]:
#@title 13. WavLM, encoder-side lookahead: 7 budgets x 2 seeds (~1 h)
!python train/train_translator.py --resume --lookahead-scope encoder_first \
    --lookaheads 0 20 40 80 160 320 640 --targets native --seeds 1337 7 \
    --steps 1200 --device cuda --out {RES}/translator_encfirst

In [ ]:
#@title 14. wav2vec 2.0 layer 2, encoder-side lookahead: 7 budgets (~35 min)
!python train/train_translator.py --resume --no-verify-causality --lookahead-scope encoder_first \
    --encoder-name facebook/wav2vec2-base --encoder-layer 2 \
    --lookaheads 0 20 40 80 160 320 640 --targets native --seeds 1337 \
    --steps 1200 --device cuda --out {RES}/w2v2_encfirst_l2

In [ ]:
#@title 15. Three schemes side by side
print(f"{'L (ms)':>7} {'per-layer':>10} {'end-to-end':>11} {'encoder-first':>14}")
enc = curve(f'{RES}/translator_encfirst.jsonl')
for L in sorted(enc):
    print(f"{L:>7g} {old.get(L, float('nan')):>10.4f} {new.get(L, float('nan')):>11.4f} {enc[L]:>14.4f}")